In [1]:
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
import os
from tqdm import tqdm

INPUT_CSV = "usa_europe_geotagged.csv"
OUT_DIR = "context_embeddings_v2"
os.makedirs(OUT_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

semantic_model = SentenceTransformer('all-MiniLM-L6-v2').to(device)

Using device: cuda


In [2]:
print("Loading dataset into memory...")

cols_to_use = ['photoid', 'latitude', 'longitude', 'datetaken', 'usertags'] 
df = pd.read_csv(INPUT_CSV, usecols=cols_to_use, low_memory=False)

df = df.dropna(subset=['latitude', 'longitude', 'datetaken'])

print(f"Total valid records ready for processing: {len(df):,}")

Loading dataset into memory...
Total valid records ready for processing: 4,927,453


In [3]:
print("Processing Cyclical Temporal Features...")

df['datetaken'] = pd.to_datetime(df['datetaken'], errors='coerce')

# Drop any rows where the date couldn't be parsed
initial_len = len(df)
df = df.dropna(subset=['datetaken'])
if len(df) < initial_len:
    print(f"Dropped {initial_len - len(df):,} rows with corrupted timestamps.")

# Extract Hour (0-23) and Month (1-12) as raw arrays for fast numpy math
hours = df['datetaken'].dt.hour.values
months = df['datetaken'].dt.month.values

# 1. Daily Cycle (24 hours)
df['hour_sin'] = np.sin(2 * np.pi * hours / 24.0).astype(np.float32)
df['hour_cos'] = np.cos(2 * np.pi * hours / 24.0).astype(np.float32)

# 2. Seasonal Cycle (12 months)
df['month_sin'] = np.sin(2 * np.pi * months / 12.0).astype(np.float32)
df['month_cos'] = np.cos(2 * np.pi * months / 12.0).astype(np.float32)

# Extract the purely temporal vector (Shape: N x 4)
temporal_vectors = df[['hour_sin', 'hour_cos', 'month_sin', 'month_cos']].values

print(f"Temporal Vectors Shape: {temporal_vectors.shape}")

Processing Cyclical Temporal Features...
Dropped 425 rows with corrupted timestamps.
Temporal Vectors Shape: (4927028, 4)


In [4]:
print("Processing 3D Spatial Embeddings...")

# Convert degrees to radians for numpy math
lat_rad = np.radians(df['latitude'].values)
lon_rad = np.radians(df['longitude'].values)

# Convert to 3D Cartesian Coordinates (Assuming Earth Radius = 1 for normalized embeddings)
df['spatial_x'] = (np.cos(lat_rad) * np.cos(lon_rad)).astype(np.float32)
df['spatial_y'] = (np.cos(lat_rad) * np.sin(lon_rad)).astype(np.float32)
df['spatial_z'] = np.sin(lat_rad).astype(np.float32)

# Extract the purely spatial vector (Shape: N x 3)
spatial_vectors = df[['spatial_x', 'spatial_y', 'spatial_z']].values

print(f"Spatial Vectors Shape: {spatial_vectors.shape}")

Processing 3D Spatial Embeddings...
Spatial Vectors Shape: (4927028, 3)


In [5]:
print("Processing Semantic Text Context...")

df['usertags'] = df['usertags'].fillna('unknown')

context_sentences = "Tags: " + df['usertags'].astype(str)

BATCH_SIZE = 4096
semantic_vectors = []

# Pass the text through the Transformer model in batches
with torch.no_grad():
    for i in tqdm(range(0, len(context_sentences), BATCH_SIZE), desc="Encoding Semantics"):
        batch = context_sentences.iloc[i:i+BATCH_SIZE].tolist()
        
        embeddings = semantic_model.encode(batch, show_progress_bar=False)
        semantic_vectors.append(embeddings)

semantic_vectors = np.vstack(semantic_vectors).astype(np.float32)

print(f"Semantic Vectors Shape: {semantic_vectors.shape}")

Processing Semantic Text Context...


Encoding Semantics: 100%|█████████████████████████████████████████████████████████████████████████████████████████████| 1203/1203 [1:13:12<00:00,  3.65s/it]


Semantic Vectors Shape: (4927028, 384)


In [9]:
print("Fusing and Normalizing Final Context Vectors...")

# 1. Concatenate along the feature dimension (columns)
# Resulting vector size: 4 (Time) + 3 (Space) + 384 (Semantics) = 391 dimensions
final_context_embeddings = np.hstack([
    temporal_vectors,
    spatial_vectors,
    semantic_vectors
])

# 2. L2 Normalization 
print("Applying L2 Normalization...")
norms = np.linalg.norm(final_context_embeddings, axis=1, keepdims=True)

norms[norms == 0] = 1e-10 
final_context_embeddings = final_context_embeddings / norms

print(f"Final Context Matrix Shape: {final_context_embeddings.shape}")

print("Saving to disk...")
np.save(os.path.join(OUT_DIR, "context_embeddings_391d.npy"), final_context_embeddings)
np.save(os.path.join(OUT_DIR, "context_photo_ids.npy"), df['photoid'].values)

print(f"\\nSuccess! State-of-the-art context embeddings saved to: {OUT_DIR}/")

Fusing and Normalizing Final Context Vectors...
Applying L2 Normalization...
Final Context Matrix Shape: (4927028, 391)
Saving to disk...
\nSuccess! State-of-the-art context embeddings saved to: context_embeddings_v2/
